In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Meeting Transcript Summarizer

## 1. Project Overview

**Task:** Convert raw meeting transcripts into structured outputs: executive summary, decisions made, blockers raised, action items with owners, and next steps.

**Approach:** Prompt decomposition — instead of one monolithic prompt, we split the extraction into focused sub-tasks that each do one thing well. We compare single-prompt vs multi-prompt pipelines.

**Stack:**
- `LangChain` + `ChatOllama` — prompt templates and LLM interaction
- `Ollama` + `qwen3.5:9b` — local LLM (no cloud API keys)

> **No API keys required.** Everything runs locally via Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Parse** unstructured meeting transcripts into speaker turns |
| 2 | **Decompose** a complex extraction task into focused sub-prompts |
| 3 | Design prompts that produce **structured JSON** reliably |
| 4 | Compare **single-prompt vs multi-prompt** pipelines on quality and reliability |
| 5 | Extract **action items** with owners, deadlines, and priorities |
| 6 | Build an **evaluation rubric** for action-item extraction |
| 7 | Format outputs for different consumers (Slack, email, Jira) |
| 8 | Handle **long transcripts** via chunking and map-reduce |

## 3. Problem Statement

After every meeting, someone has to write up:
- What was discussed (summary)
- What was decided (decisions)
- What's blocking progress (blockers)
- Who owns what next (action items)
- What happens next (next steps)

This takes 15–30 minutes per meeting and is often done poorly or not at all. An LLM can do it in seconds — but only with well-designed prompts.

## 4. Why This Project Matters

- **Massive market:** Otter.ai, Fireflies.ai, and Microsoft Copilot all offer meeting summarization
- **Prompt decomposition** is the core technique — breaking a complex task into sub-tasks is how you build reliable LLM applications
- **Structured extraction** (getting JSON from prose) is a transferable skill used in RAG, agents, and data pipelines
- **Evaluation of extraction** teaches you how to measure LLM quality beyond simple accuracy

## 5. Pipeline Architecture

```
Raw Transcript
      │
      ▼
  [Parse] ── speaker identification, turn segmentation, timestamp handling
      │
      ▼
  [Decomposed Extraction]
      │
      ├──▶ Prompt 1: Executive Summary (3–5 sentences)
      ├──▶ Prompt 2: Decisions Made (explicit agreements)
      ├──▶ Prompt 3: Blockers & Risks (unresolved issues)
      ├──▶ Prompt 4: Action Items (who, what, when)
      └──▶ Prompt 5: Next Steps (future plans)
      │
      ▼
  [Format] ── Markdown / JSON / Slack / Email
      │
      ▼
  [Evaluate] ── score action-item extraction vs ground truth
```

### Why Decompose?

| Single Prompt | Decomposed Prompts |
|---------------|--------------------|
| One LLM call handles everything | Separate call per extraction type |
| Cheaper and faster | More reliable — each prompt is focused |
| Often misses categories or mixes them | Won't skip a section (it's a separate call) |
| Hard to debug or improve one part | Fix one prompt without affecting others |
| Output format often breaks | Smaller JSON structures are easier for LLMs |

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core

print("Dependencies: langchain, langchain-ollama")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports & Helpers

In [ ]:
import os
import json
import re
import warnings
import time
from textwrap import dedent

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

print("All imports OK")

In [ ]:
LLM_MODEL = "qwen3.5:9b"

llm = ChatOllama(model=LLM_MODEL, temperature=0)


def clean(text: str) -> str:
    """Strip thinking tags from model output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str) -> dict | list | None:
    """Extract JSON (object or array) from LLM output."""
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    # Try object
    start = text.find("{")
    end = text.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    # Try array
    start = text.find("[")
    end = text.rfind("]") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(system: str, user: str) -> str:
    """Send a prompt and return cleaned text."""
    resp = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
    return clean(resp.content)


# Quick test
test = ask("Reply with one word.", "Say ready.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Transcript Data & Parsing

## 8. Sample Meeting Transcripts

We create three realistic transcripts of varying complexity. Each has ground-truth annotations so we can evaluate extraction quality.

In [ ]:
TRANSCRIPTS = {
    # ── Transcript 1: Sprint Planning ─────────────────────
    "sprint_planning": {
        "title": "Sprint 14 Planning — Backend Team",
        "date": "2026-04-06",
        "attendees": ["Sarah (Engineering Manager)", "Dev (Senior Backend)", "Priya (Backend)", "Tom (QA)"],
        "transcript": dedent("""\
            Sarah: Alright everyone, let's kick off Sprint 14 planning. Last sprint we closed 18 out of 22 story points. Four tickets rolled over — Dev, can you give an update on those?

            Dev: Yeah, the authentication refactor is about 80% done. I hit a blocker with the OAuth provider — their sandbox environment has been down for three days. I've filed a support ticket with them but no response yet. The other rollover was the database migration script, which I finished Friday.

            Sarah: OK, so the database migration is done. For the OAuth blocker, let's not wait any longer. Dev, can you implement a mock OAuth provider for testing so we're not blocked? We can switch to the real one when their sandbox is back.

            Dev: That makes sense. I can have the mock ready by Wednesday.

            Sarah: Great. Priya, what's the status on the payment integration?

            Priya: The Stripe integration is working in dev. I need to add error handling for declined cards and webhook retry logic. I also noticed our current retry logic doesn't handle idempotency keys correctly — we could end up charging customers twice if a webhook fires during a retry.

            Sarah: That's a serious issue. Let's make fixing the idempotency problem the top priority. Priya, can you write up the bug and we'll fast-track it?

            Priya: Sure, I'll create the ticket today and start on the fix tomorrow. I think it's a two-day effort.

            Sarah: Tom, how's the test coverage looking?

            Tom: We're at 72% coverage on the backend, up from 65% last sprint. I want to push to 80% this sprint. I've identified the biggest gaps — the payment module and the notification service. I'll need Priya's help writing test fixtures for the Stripe mock.

            Priya: Happy to help. Let's pair on that Thursday afternoon.

            Tom: Works for me.

            Sarah: One more thing — leadership wants a demo of the payment flow by next Friday. Dev, Priya, and Tom — can you coordinate on having a working demo in staging by Thursday so we can do a dry run?

            Dev: I'll make sure the auth piece doesn't block the demo. Staging deployment should be straightforward.

            Priya: I'll have the payment flow ready. The idempotency fix will be in by then.

            Tom: I'll write end-to-end tests for the demo path by Wednesday.

            Sarah: Perfect. Let's plan the dry run for Thursday at 3pm. I'll send the invite. Anything else?

            Dev: Quick thing — our staging database is running out of disk space. We should request a capacity increase before the demo.

            Sarah: Good catch. Dev, can you file that infra request today?

            Dev: Will do.

            Sarah: Alright, that's a wrap. Thanks everyone."""),

        "ground_truth": {
            "decisions": [
                "Implement mock OAuth provider instead of waiting for sandbox",
                "Fast-track idempotency key bug fix as top priority",
                "Schedule demo dry run for Thursday at 3pm",
            ],
            "blockers": [
                "OAuth provider sandbox has been down for 3 days, no response from support",
                "Staging database running out of disk space",
            ],
            "action_items": [
                {"owner": "Dev", "task": "Build mock OAuth provider", "deadline": "Wednesday"},
                {"owner": "Priya", "task": "Create idempotency bug ticket", "deadline": "today"},
                {"owner": "Priya", "task": "Fix idempotency key issue", "deadline": "2 days"},
                {"owner": "Tom", "task": "Write end-to-end tests for demo path", "deadline": "Wednesday"},
                {"owner": "Priya & Tom", "task": "Pair on Stripe test fixtures", "deadline": "Thursday afternoon"},
                {"owner": "Dev", "task": "File infra request for staging disk space", "deadline": "today"},
                {"owner": "Sarah", "task": "Send dry run calendar invite", "deadline": "today"},
            ],
        },
    },

    # ── Transcript 2: Product Review ──────────────────────
    "product_review": {
        "title": "Q2 Product Review — Mobile App",
        "date": "2026-04-08",
        "attendees": ["Alex (Product Manager)", "Jordan (Design Lead)", "Kim (iOS Dev)", "Ravi (Analytics)"],
        "transcript": dedent("""\
            Alex: Let's review where we are on the mobile app for Q2. Ravi, can you share the numbers?

            Ravi: Sure. DAU is at 45K, up 12% from last quarter. Retention at day-7 dropped from 38% to 34% though. The biggest drop-off is during onboarding — we're losing 40% of users before they complete profile setup.

            Alex: That's concerning. Jordan, you had some ideas about simplifying onboarding?

            Jordan: Yes, I've been working on a streamlined flow. Instead of the current 6-step onboarding, we can reduce it to 3 steps by deferring non-essential profile fields to later. I have mockups ready — I'll share them in Figma today. The key change is making profile photo and bio optional during initial signup.

            Alex: I like that. Let's do it. Kim, how long would the implementation take?

            Kim: If Jordan's mockups are final, probably 5 days for iOS. But I'm currently blocked on the push notification refactor — Apple rejected our last build because of the notification permission flow. I need to fix that first, which is about 2 days.

            Alex: OK, prioritize the Apple rejection fix first. That's blocking our next release. Then move to onboarding.

            Ravi: One more data point — users who enable push notifications have 2x higher day-30 retention. So the notification flow is actually critical for the metrics we care about.

            Alex: Great insight, Ravi. Can you put together a one-pager on notification opt-in rates across different prompt timings? That would help us decide when to ask for permission.

            Ravi: I'll have that by end of week.

            Jordan: I also want to flag — our design system components are getting outdated. Several screens are using deprecated patterns. I'd like to propose a design debt sprint next quarter.

            Alex: Noted. Let's add that to the Q3 planning backlog. For now, focus on onboarding.

            Kim: One risk — the new onboarding flow might break our analytics events. Ravi, can you check if the current event schema can handle a 3-step flow?

            Ravi: Good point. I'll audit the event schema and flag any issues by Tuesday.

            Alex: Great. Let's regroup next Wednesday to review Jordan's mockups and Ravi's notification analysis. Kim, by then hopefully the Apple fix is shipped.

            Kim: Should be. I'll submit the new build by Monday.

            Alex: Perfect. Thanks everyone."""),

        "ground_truth": {
            "decisions": [
                "Reduce onboarding from 6 steps to 3 by deferring non-essential fields",
                "Prioritize Apple rejection fix before onboarding redesign",
                "Add design debt sprint to Q3 planning backlog",
            ],
            "blockers": [
                "Apple rejected build due to notification permission flow",
                "New onboarding flow may break analytics event schema",
            ],
            "action_items": [
                {"owner": "Jordan", "task": "Share onboarding mockups in Figma", "deadline": "today"},
                {"owner": "Kim", "task": "Fix Apple notification permission rejection", "deadline": "2 days"},
                {"owner": "Kim", "task": "Submit new build to Apple", "deadline": "Monday"},
                {"owner": "Ravi", "task": "Create notification opt-in timing analysis one-pager", "deadline": "end of week"},
                {"owner": "Ravi", "task": "Audit event schema for 3-step onboarding compatibility", "deadline": "Tuesday"},
            ],
        },
    },

    # ── Transcript 3: Incident Postmortem ─────────────────
    "postmortem": {
        "title": "Incident Postmortem — Payment Service Outage (April 3)",
        "date": "2026-04-09",
        "attendees": ["Lisa (VP Eng)", "Marcus (SRE Lead)", "Chen (Payments Team)", "Fatima (On-call Eng)"],
        "transcript": dedent("""\
            Lisa: Let's go through the April 3rd incident. Marcus, can you walk us through the timeline?

            Marcus: At 14:23 UTC we got paged — payment processing latency spiked from 200ms to 12 seconds. By 14:30 it was effectively a full outage. Fatima was on call and responded within 2 minutes.

            Fatima: When I got the page, I checked the dashboard and saw the payment service pods were OOMKilled. Memory usage had spiked to 4x normal. I tried scaling up the pods but the new pods kept crashing too. I escalated to Marcus at 14:40.

            Marcus: I looked at the deploy history and found that a config change went out at 14:15 — it doubled the batch size for payment reconciliation from 500 to 1000. That was the trigger. The reconciliation job was loading twice as much data into memory, which caused the OOM kills.

            Chen: That config change was mine. I was trying to speed up the reconciliation job because finance complained it was too slow. I tested it in staging but staging only has 10% of production data, so the memory spike didn't show up.

            Lisa: How did we recover?

            Marcus: We rolled back the config at 15:05. Services recovered by 15:12. Total outage duration was 42 minutes. We estimate about 3,200 transactions failed during the window. No data was lost — the transactions queued and processed after recovery.

            Lisa: OK. Let's talk about what we need to fix. First — why didn't our staging tests catch this?

            Chen: Staging has a tiny dataset. We've been saying we need production-like data in staging for months but it hasn't been prioritized.

            Marcus: Agreed. We also need config change reviews for anything touching batch sizes or resource limits. Right now anyone can push a config change without review.

            Lisa: Let's make that a hard requirement. Any config change that touches resource parameters needs a senior engineer review. Marcus, can you update the deploy policy by end of week?

            Marcus: Yes, I'll draft a new policy and circulate it for review.

            Fatima: I also want to flag — our alerting didn't catch the memory spike early enough. We got paged on latency, not on memory. If we had memory-based alerts, we could have caught it 5 minutes earlier.

            Marcus: Good point. Fatima, can you set up memory-based PagerDuty alerts for all payment service pods? Let's use 80% memory threshold.

            Fatima: I'll have that done by Tuesday.

            Lisa: Chen, for the reconciliation performance issue that finance raised — what's the right fix that doesn't involve increasing batch size?

            Chen: We should switch to streaming processing instead of batch loading. That's a bigger change — probably a two-week project. I'll write up a proposal.

            Lisa: Do that. Let's also send a customer impact report to the account management team. Marcus, can you draft that today?

            Marcus: Will do.

            Lisa: Last thing — let's schedule a follow-up in two weeks to check that the action items are done. Any other issues?

            Fatima: Just one — our runbook for payment service incidents is outdated. It doesn't mention the config rollback procedure. I'll update it this week.

            Lisa: Good. Thank you all for a thorough postmortem."""),

        "ground_truth": {
            "decisions": [
                "Config changes touching resource parameters now require senior engineer review",
                "Switch reconciliation from batch to streaming processing",
                "Set 80% memory threshold for payment service alerts",
            ],
            "blockers": [
                "Staging environment lacks production-like data volume",
                "No config change review process for resource parameters",
                "Memory-based alerting not set up for payment services",
            ],
            "action_items": [
                {"owner": "Marcus", "task": "Draft updated deploy policy for config changes", "deadline": "end of week"},
                {"owner": "Fatima", "task": "Set up memory-based PagerDuty alerts (80% threshold)", "deadline": "Tuesday"},
                {"owner": "Chen", "task": "Write proposal for streaming reconciliation", "deadline": "unspecified"},
                {"owner": "Marcus", "task": "Draft customer impact report for account management", "deadline": "today"},
                {"owner": "Fatima", "task": "Update payment service incident runbook", "deadline": "this week"},
                {"owner": "Lisa", "task": "Schedule 2-week follow-up meeting", "deadline": "unspecified"},
            ],
        },
    },
}

print(f"Loaded {len(TRANSCRIPTS)} transcripts:")
for key, t in TRANSCRIPTS.items():
    lines = t["transcript"].strip().split("\n")
    turns = sum(1 for l in lines if l and ":" in l[:30])
    gt = t["ground_truth"]
    print(f"  {key:20s} | {len(t['attendees'])} attendees | ~{turns} turns | "
          f"{len(gt['decisions'])} decisions | {len(gt['action_items'])} actions")

## 9. Transcript Parsing

Before sending text to the LLM, we parse the raw transcript into structured speaker turns. This lets us:
- Count participation per speaker
- Chunk long transcripts by turn
- Show the LLM cleaner input

In [ ]:
def parse_transcript(text: str) -> list[dict]:
    """
    Parse 'Speaker: text' format into structured turns.
    Handles multi-line turns (continuation lines without a speaker prefix).
    """
    turns = []
    current_speaker = None
    current_text = []

    for line in text.strip().split("\n"):
        line = line.strip()
        if not line:
            continue

        # Check for speaker prefix — word(s) followed by colon
        match = re.match(r"^([A-Za-z]+):\s*(.*)$", line)
        if match:
            # Save previous turn
            if current_speaker and current_text:
                turns.append({"speaker": current_speaker, "text": " ".join(current_text)})
            current_speaker = match.group(1)
            current_text = [match.group(2)] if match.group(2) else []
        else:
            # Continuation of previous turn
            if current_speaker:
                current_text.append(line)

    # Save last turn
    if current_speaker and current_text:
        turns.append({"speaker": current_speaker, "text": " ".join(current_text)})

    return turns


# Parse and inspect the first transcript
t1 = TRANSCRIPTS["sprint_planning"]
turns = parse_transcript(t1["transcript"])

print(f"Parsed {len(turns)} speaker turns from '{t1['title']}'\n")
for i, turn in enumerate(turns[:5]):
    print(f"  Turn {i+1} [{turn['speaker']:6s}]: {turn['text'][:80]}...")

# Speaker participation stats
from collections import Counter
speaker_counts = Counter(t["speaker"] for t in turns)
print(f"\nSpeaker participation:")
for speaker, count in speaker_counts.most_common():
    total_words = sum(len(t["text"].split()) for t in turns if t["speaker"] == speaker)
    print(f"  {speaker:8s}: {count} turns, {total_words} words")

---

# Part B — Prompt Decomposition

## 10. Approach 1: Single Monolithic Prompt

First, we try the naive approach — one prompt that extracts everything at once. This establishes a baseline.

In [ ]:
MONOLITHIC_SYSTEM = """You are a meeting notes assistant. Analyze meeting transcripts and extract
structured information. Return ONLY valid JSON. No explanation."""

MONOLITHIC_USER = """Analyze this meeting transcript and extract ALL of the following:

1. Executive summary (3-5 sentences)
2. Key decisions made
3. Blockers and risks identified
4. Action items with owners and deadlines
5. Next steps

Meeting: {title}
Date: {date}
Attendees: {attendees}

Transcript:
{transcript}

Return JSON:
{{
  "summary": "3-5 sentence executive summary",
  "decisions": ["decision 1", "decision 2"],
  "blockers": ["blocker 1", "blocker 2"],
  "action_items": [
    {{"owner": "Name", "task": "description", "deadline": "when"}}
  ],
  "next_steps": ["step 1", "step 2"]
}}"""


def run_monolithic(key: str) -> dict:
    """Run single-prompt extraction on a transcript."""
    t = TRANSCRIPTS[key]
    t0 = time.perf_counter()
    resp = ask(
        MONOLITHIC_SYSTEM,
        MONOLITHIC_USER.format(
            title=t["title"],
            date=t["date"],
            attendees=", ".join(t["attendees"]),
            transcript=t["transcript"],
        ),
    )
    elapsed = time.perf_counter() - t0
    parsed = parse_json(resp)
    if parsed is None:
        parsed = {"error": "Failed to parse JSON", "raw": resp[:500]}
    parsed["_time_sec"] = round(elapsed, 1)
    return parsed


# Run on the sprint planning transcript
print("MONOLITHIC EXTRACTION — Sprint Planning")
print("=" * 60)
mono_result = run_monolithic("sprint_planning")
print(json.dumps(mono_result, indent=2))

## 11. Approach 2: Decomposed Prompts

Now the better approach — each extraction type gets its own focused prompt. Each prompt:
- Has a narrow, clear task
- Returns a small, predictable JSON structure
- Can be debugged and improved independently

### Prompt 1: Executive Summary

In [ ]:
SUMMARY_SYSTEM = """You write concise executive summaries of meetings.
Rules:
- Write exactly 3-5 sentences
- Focus on outcomes, not blow-by-blow narration
- Mention the most important decision, biggest risk, and key owners
- Write for someone who wasn't in the meeting and needs to know what happened in 30 seconds
- Return ONLY the summary text, no JSON, no headers"""

SUMMARY_USER = """Meeting: {title}
Date: {date}
Attendees: {attendees}

Transcript:
{transcript}

Executive summary:"""

print("Summary prompt defined")
print(f"System: {len(SUMMARY_SYSTEM)} chars")

### Prompt 2: Decisions

In [ ]:
DECISIONS_SYSTEM = """You extract decisions from meeting transcripts.

A "decision" is an explicit agreement or resolution. It must be:
- Something the group AGREED to do (not just discussed)
- A clear direction or choice ("Let's do X" or "We decided Y")
- NOT a task assignment (those are action items, not decisions)

Return JSON: {"decisions": ["decision 1", "decision 2"]}
If no decisions, return {"decisions": []}"""

DECISIONS_USER = """Meeting: {title}

Transcript:
{transcript}

Extract all decisions:"""

print("Decisions prompt defined")

### Prompt 3: Blockers & Risks

In [ ]:
BLOCKERS_SYSTEM = """You extract blockers and risks from meeting transcripts.

A "blocker" is something currently preventing progress.
A "risk" is something that COULD cause problems if not addressed.

For each, identify:
- What the issue is
- Who or what is affected
- Whether it's a current blocker or a future risk

Return JSON:
{"blockers": [
  {"issue": "description", "affected": "who/what", "type": "blocker|risk"}
]}"""

BLOCKERS_USER = """Meeting: {title}

Transcript:
{transcript}

Extract all blockers and risks:"""

print("Blockers prompt defined")

### Prompt 4: Action Items

In [ ]:
ACTIONS_SYSTEM = """You extract action items from meeting transcripts.

An action item is a specific task that a specific person committed to doing.
It must have:
- An OWNER (the person who will do it — use their first name)
- A TASK (what they committed to)
- A DEADLINE (when they said they'd do it — use "unspecified" if none given)
- A PRIORITY (high/medium/low — based on urgency language in the transcript)

Rules:
- Only include items with a clear owner
- Do NOT include vague intentions ("we should think about X")
- DO include items where someone said "I'll do X" or was directly asked and agreed

Return JSON:
{"action_items": [
  {"owner": "Name", "task": "description", "deadline": "when", "priority": "high|medium|low"}
]}"""

ACTIONS_USER = """Meeting: {title}

Transcript:
{transcript}

Extract all action items:"""

print("Action items prompt defined")

### Prompt 5: Next Steps

In [ ]:
NEXTSTEPS_SYSTEM = """You extract the next steps and follow-ups from meeting transcripts.

Next steps are future plans, follow-up meetings, milestones, or checkpoints.
They differ from action items — they're about what happens NEXT in the project timeline,
not individual tasks.

Return JSON: {"next_steps": ["step 1", "step 2"]}"""

NEXTSTEPS_USER = """Meeting: {title}

Transcript:
{transcript}

What are the next steps?"""

print("Next steps prompt defined")

## 12. Run Decomposed Pipeline

In [ ]:
def run_decomposed(key: str) -> dict:
    """Run decomposed multi-prompt extraction on a transcript."""
    t = TRANSCRIPTS[key]
    common = {
        "title": t["title"],
        "date": t["date"],
        "attendees": ", ".join(t["attendees"]),
        "transcript": t["transcript"],
    }

    results = {}
    t0 = time.perf_counter()

    # 1. Summary (returns text, not JSON)
    results["summary"] = ask(SUMMARY_SYSTEM, SUMMARY_USER.format(**common))

    # 2. Decisions
    resp = ask(DECISIONS_SYSTEM, DECISIONS_USER.format(**common))
    parsed = parse_json(resp)
    results["decisions"] = parsed.get("decisions", []) if parsed else []

    # 3. Blockers
    resp = ask(BLOCKERS_SYSTEM, BLOCKERS_USER.format(**common))
    parsed = parse_json(resp)
    results["blockers"] = parsed.get("blockers", []) if parsed else []

    # 4. Action items
    resp = ask(ACTIONS_SYSTEM, ACTIONS_USER.format(**common))
    parsed = parse_json(resp)
    results["action_items"] = parsed.get("action_items", []) if parsed else []

    # 5. Next steps
    resp = ask(NEXTSTEPS_SYSTEM, NEXTSTEPS_USER.format(**common))
    parsed = parse_json(resp)
    results["next_steps"] = parsed.get("next_steps", []) if parsed else []

    results["_time_sec"] = round(time.perf_counter() - t0, 1)
    return results


# Run on sprint planning
print("DECOMPOSED EXTRACTION — Sprint Planning")
print("=" * 60)
decomp_result = run_decomposed("sprint_planning")

print(f"\nSummary:\n{decomp_result['summary']}")
print(f"\nDecisions ({len(decomp_result['decisions'])}):\n" +
      "\n".join(f"  • {d}" for d in decomp_result["decisions"]))
print(f"\nBlockers ({len(decomp_result['blockers'])}):\n" +
      "\n".join(f"  • {b['issue'] if isinstance(b, dict) else b}" for b in decomp_result["blockers"]))
print(f"\nAction Items ({len(decomp_result['action_items'])}):")
for a in decomp_result["action_items"]:
    if isinstance(a, dict):
        print(f"  • [{a.get('owner','?'):8s}] {a.get('task','?')} — by {a.get('deadline','?')}")
    else:
        print(f"  • {a}")
print(f"\nNext Steps ({len(decomp_result['next_steps'])}):\n" +
      "\n".join(f"  • {s}" for s in decomp_result["next_steps"]))
print(f"\nTotal time: {decomp_result['_time_sec']}s")

## 13. Run on All Transcripts

Process all three transcripts and store results for comparison.

In [ ]:
all_mono = {}
all_decomp = {}

for key in TRANSCRIPTS:
    t = TRANSCRIPTS[key]
    print(f"\nProcessing: {t['title']}")
    print("-" * 50)

    # Monolithic
    mono = run_monolithic(key)
    all_mono[key] = mono
    print(f"  Monolithic:  {mono.get('_time_sec', '?')}s | "
          f"{len(mono.get('decisions', []))} decisions | "
          f"{len(mono.get('action_items', []))} actions")

    # Decomposed
    decomp = run_decomposed(key)
    all_decomp[key] = decomp
    print(f"  Decomposed:  {decomp['_time_sec']}s | "
          f"{len(decomp['decisions'])} decisions | "
          f"{len(decomp['action_items'])} actions")

print("\nAll transcripts processed!")

---

# Part C — Evaluation & Comparison

## 14. Monolithic vs Decomposed — Quantitative Comparison

In [ ]:
print("MONOLITHIC vs DECOMPOSED COMPARISON")
print("=" * 70)

for key in TRANSCRIPTS:
    gt = TRANSCRIPTS[key]["ground_truth"]
    mono = all_mono[key]
    decomp = all_decomp[key]

    print(f"\n{TRANSCRIPTS[key]['title']}")
    print("-" * 50)

    for field in ["decisions", "action_items"]:
        gt_count = len(gt.get(field, []))
        mono_count = len(mono.get(field, []))
        decomp_count = len(decomp.get(field, []))
        print(f"  {field:15s}: GT={gt_count:2d} | Mono={mono_count:2d} | Decomp={decomp_count:2d}")

    mono_t = mono.get("_time_sec", 0)
    decomp_t = decomp.get("_time_sec", 0)
    print(f"  {'time':15s}: Mono={mono_t:.1f}s | Decomp={decomp_t:.1f}s")

## 15. Action-Item Extraction Evaluation

How do we measure whether the extracted action items are correct? We use a **fuzzy matching** approach:

- For each ground-truth item, check if any extracted item matches on **owner** AND **task content**
- **Precision** = how many extracted items are correct
- **Recall** = how many ground-truth items were found
- **F1** = harmonic mean of precision and recall

In [ ]:
def normalize(text: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace."""
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return re.sub(r"\s+", " ", text)


def fuzzy_match_action(gt_item: dict, pred_item: dict, overlap_threshold: float = 0.4) -> bool:
    """
    Check if a predicted action item roughly matches a ground-truth item.
    Matches on: (1) same owner AND (2) enough keyword overlap in the task.
    """
    # Owner match (case-insensitive, handle "Priya & Tom" vs "Priya")
    gt_owner = normalize(gt_item.get("owner", ""))
    pred_owner = normalize(pred_item.get("owner", ""))
    owner_match = any(
        name in pred_owner
        for name in gt_owner.split(" ")
        if len(name) > 2
    )
    if not owner_match:
        return False

    # Task keyword overlap
    gt_words = set(normalize(gt_item.get("task", "")).split())
    pred_words = set(normalize(pred_item.get("task", "")).split())
    if not gt_words:
        return False
    overlap = len(gt_words & pred_words) / len(gt_words)
    return overlap >= overlap_threshold


def evaluate_actions(gt_items: list[dict], pred_items: list[dict]) -> dict:
    """Compute precision, recall, F1 for action-item extraction."""
    matched_gt = set()
    matched_pred = set()

    for gi, gt in enumerate(gt_items):
        for pi, pred in enumerate(pred_items):
            if isinstance(pred, dict) and fuzzy_match_action(gt, pred):
                matched_gt.add(gi)
                matched_pred.add(pi)
                break

    tp = len(matched_gt)
    precision = tp / len(pred_items) if pred_items else 0
    recall = tp / len(gt_items) if gt_items else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "true_positives": tp,
        "predicted": len(pred_items),
        "ground_truth": len(gt_items),
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "missed": [gt_items[i] for i in range(len(gt_items)) if i not in matched_gt],
    }


print("Evaluation functions defined")

In [ ]:
print("ACTION-ITEM EXTRACTION EVALUATION")
print("=" * 70)

for key in TRANSCRIPTS:
    gt_items = TRANSCRIPTS[key]["ground_truth"]["action_items"]

    # Evaluate monolithic
    mono_actions = all_mono[key].get("action_items", [])
    mono_eval = evaluate_actions(gt_items, mono_actions)

    # Evaluate decomposed
    decomp_actions = all_decomp[key].get("action_items", [])
    decomp_eval = evaluate_actions(gt_items, decomp_actions)

    print(f"\n{TRANSCRIPTS[key]['title']}")
    print(f"  Ground truth: {len(gt_items)} action items")
    print(f"  Monolithic:  P={mono_eval['precision']:.2f} R={mono_eval['recall']:.2f} F1={mono_eval['f1']:.2f} "
          f"({mono_eval['true_positives']}/{mono_eval['ground_truth']} found)")
    print(f"  Decomposed:  P={decomp_eval['precision']:.2f} R={decomp_eval['recall']:.2f} F1={decomp_eval['f1']:.2f} "
          f"({decomp_eval['true_positives']}/{decomp_eval['ground_truth']} found)")

    if decomp_eval["missed"]:
        print(f"  Missed by decomposed:")
        for m in decomp_eval["missed"]:
            print(f"    - [{m['owner']}] {m['task']}")

---

# Part D — Output Formatting

## 16. Format as Markdown Meeting Notes

In [ ]:
def format_markdown(title: str, date: str, attendees: list[str], result: dict) -> str:
    """Format extraction results as clean Markdown meeting notes."""
    lines = [
        f"# {title}",
        f"**Date:** {date}  ",
        f"**Attendees:** {', '.join(attendees)}\n",
        "## Executive Summary",
        result.get("summary", "(no summary)"),
        "",
        "## Decisions",
    ]
    for d in result.get("decisions", []):
        lines.append(f"- ✅ {d}")

    lines.append("\n## Blockers & Risks")
    for b in result.get("blockers", []):
        if isinstance(b, dict):
            icon = "🔴" if b.get("type") == "blocker" else "🟡"
            lines.append(f"- {icon} {b['issue']} (affects: {b.get('affected', '?')})")
        else:
            lines.append(f"- 🔴 {b}")

    lines.append("\n## Action Items")
    lines.append("| Owner | Task | Deadline | Priority |")
    lines.append("|-------|------|----------|----------|")
    for a in result.get("action_items", []):
        if isinstance(a, dict):
            lines.append(f"| {a.get('owner','?')} | {a.get('task','?')} | "
                         f"{a.get('deadline','?')} | {a.get('priority','medium')} |")

    lines.append("\n## Next Steps")
    for s in result.get("next_steps", []):
        lines.append(f"- {s}")

    return "\n".join(lines)


# Format the sprint planning result
t = TRANSCRIPTS["sprint_planning"]
md = format_markdown(t["title"], t["date"], t["attendees"], all_decomp["sprint_planning"])
print(md)

## 17. Format as Slack Message

In [ ]:
def format_slack(title: str, result: dict) -> str:
    """Format results as a Slack-friendly message."""
    lines = [
        f"*📋 Meeting Notes: {title}*",
        "",
        f"*Summary:* {result.get('summary', 'N/A')}",
        "",
    ]

    decisions = result.get("decisions", [])
    if decisions:
        lines.append(f"*✅ Decisions ({len(decisions)}):*")
        for d in decisions:
            lines.append(f"  • {d}")
        lines.append("")

    blockers = result.get("blockers", [])
    if blockers:
        lines.append(f"*🚧 Blockers ({len(blockers)}):*")
        for b in blockers:
            issue = b["issue"] if isinstance(b, dict) else b
            lines.append(f"  • {issue}")
        lines.append("")

    actions = result.get("action_items", [])
    if actions:
        lines.append(f"*📌 Action Items ({len(actions)}):*")
        for a in actions:
            if isinstance(a, dict):
                lines.append(f"  • *{a.get('owner','?')}*: {a.get('task','?')} _(by {a.get('deadline','?')})_")
        lines.append("")

    return "\n".join(lines)


print(format_slack(t["title"], all_decomp["sprint_planning"]))

## 18. Format as JSON Export

Structured JSON for integration with project management tools (Jira, Linear, Asana).

In [ ]:
def format_json_export(key: str, result: dict) -> dict:
    """Create a structured JSON export for tool integrations."""
    t = TRANSCRIPTS[key]
    return {
        "meeting": {
            "title": t["title"],
            "date": t["date"],
            "attendees": t["attendees"],
        },
        "summary": result.get("summary", ""),
        "decisions": result.get("decisions", []),
        "blockers": result.get("blockers", []),
        "action_items": result.get("action_items", []),
        "next_steps": result.get("next_steps", []),
        "metadata": {
            "extraction_time_sec": result.get("_time_sec", 0),
            "model": LLM_MODEL,
            "method": "decomposed",
        },
    }


export = format_json_export("sprint_planning", all_decomp["sprint_planning"])
print(json.dumps(export, indent=2))

---

# Part E — Handling Long Transcripts

## 19. Chunked Processing for Long Meetings

Real meetings can produce 5,000–50,000 word transcripts that exceed context windows. We solve this with **map-reduce**: extract from each chunk, then merge results.

In [ ]:
def chunk_transcript(text: str, max_turns: int = 15) -> list[str]:
    """
    Split a transcript into chunks of ~max_turns speaker turns.
    Never splits mid-turn.
    """
    turns = parse_transcript(text)
    chunks = []
    for i in range(0, len(turns), max_turns):
        chunk_turns = turns[i:i + max_turns]
        chunk_text = "\n\n".join(f"{t['speaker']}: {t['text']}" for t in chunk_turns)
        chunks.append(chunk_text)
    return chunks


MERGE_SYSTEM = """You merge and deduplicate lists of extracted meeting information.
Combine the partial results below into a single clean list.
Remove exact or near-duplicates. Keep the most specific version of each item.
Return JSON."""

MERGE_USER = """Merge these partial action item lists into one deduplicated list.

Partial lists:
{partial_json}

Return: {{"action_items": [...]}}"""


def extract_chunked(key: str, max_turns: int = 15) -> dict:
    """Extract from a transcript using chunked map-reduce."""
    t = TRANSCRIPTS[key]
    chunks = chunk_transcript(t["transcript"], max_turns)
    print(f"  Split into {len(chunks)} chunks")

    # Map: extract action items from each chunk
    all_actions = []
    for i, chunk in enumerate(chunks):
        resp = ask(ACTIONS_SYSTEM, ACTIONS_USER.format(title=t["title"], transcript=chunk))
        parsed = parse_json(resp)
        items = parsed.get("action_items", []) if parsed else []
        all_actions.extend(items)
        print(f"    Chunk {i+1}: {len(items)} action items")

    # Reduce: merge and deduplicate
    if len(chunks) > 1 and all_actions:
        resp = ask(MERGE_SYSTEM, MERGE_USER.format(partial_json=json.dumps(all_actions, indent=2)))
        parsed = parse_json(resp)
        merged = parsed.get("action_items", all_actions) if parsed else all_actions
    else:
        merged = all_actions

    return {"action_items": merged, "chunks": len(chunks)}


# Demo: chunk the postmortem transcript (will be small chunks for demo)
print("CHUNKED EXTRACTION — Postmortem")
print("=" * 50)
chunked = extract_chunked("postmortem", max_turns=8)
print(f"\nMerged result: {len(chunked['action_items'])} action items")
for a in chunked["action_items"]:
    if isinstance(a, dict):
        print(f"  • [{a.get('owner','?')}] {a.get('task','?')}")

## 20. Full Pipeline Function

In [ ]:
def summarize_meeting(key: str, output_format: str = "markdown") -> str:
    """
    End-to-end: parse → extract (decomposed) → format.

    output_format: 'markdown', 'slack', or 'json'
    """
    t = TRANSCRIPTS[key]

    # Extract
    result = run_decomposed(key)

    # Format
    if output_format == "markdown":
        return format_markdown(t["title"], t["date"], t["attendees"], result)
    elif output_format == "slack":
        return format_slack(t["title"], result)
    elif output_format == "json":
        return json.dumps(format_json_export(key, result), indent=2)
    else:
        return str(result)


# Generate all three formats for the product review
for fmt in ["markdown", "slack"]:
    print(f"\n{'=' * 60}")
    print(f"OUTPUT FORMAT: {fmt.upper()}")
    print("=" * 60)
    output = summarize_meeting("product_review", output_format=fmt)
    print(output)

---

## 21. Error Analysis — Edge Cases

In [ ]:
# ── Edge case 1: Implicit action item (no explicit commitment) ──
print("EDGE CASE 1: Implicit vs explicit action items")
print("-" * 50)

implicit_transcript = dedent("""\
    Alice: We really should update our documentation.
    Bob: Yeah, it's been outdated for months.
    Alice: Someone should probably look into that.
    Bob: Agreed. Also, I'll set up the new CI pipeline by Thursday.""")

resp = ask(ACTIONS_SYSTEM, ACTIONS_USER.format(
    title="Quick Sync", transcript=implicit_transcript
))
parsed = parse_json(resp)
actions = parsed.get("action_items", []) if parsed else []
print(f"  Extracted: {len(actions)} action items")
for a in actions:
    if isinstance(a, dict):
        print(f"    [{a.get('owner','?')}] {a.get('task','?')}")
print("  → Should find Bob's CI pipeline task but NOT the vague 'someone should update docs'")

# ── Edge case 2: Decision disguised as a question ──────
print(f"\nEDGE CASE 2: Decision disguised as a question")
print("-" * 50)

question_transcript = dedent("""\
    Manager: Should we go with vendor A or vendor B?
    Lead: Vendor A is cheaper but vendor B has better support.
    Manager: Let's go with vendor B then. The support is worth the extra cost.
    Lead: Makes sense.""")

resp = ask(DECISIONS_SYSTEM, DECISIONS_USER.format(
    title="Vendor Selection", transcript=question_transcript
))
parsed = parse_json(resp)
decisions = parsed.get("decisions", []) if parsed else []
print(f"  Extracted decisions: {decisions}")
print("  → Should find 'Go with vendor B' despite it starting as a question")

# ── Edge case 3: Noisy transcript with filler ──────────
print(f"\nEDGE CASE 3: Very noisy transcript")
print("-" * 50)

noisy_transcript = dedent("""\
    Alice: Um, so, uh, I think we need to, like, move the deadline? For the, you know, the API thing?
    Bob: Yeah yeah yeah, totally. So basically what I was thinking is, right, we push it to next Friday instead of this Wednesday?
    Alice: That works. Oh and, um, can you also send me those wireframes? The ones from, uh, last week's session?
    Bob: Sure sure, I'll get those to you by end of day.""")

resp = ask(ACTIONS_SYSTEM, ACTIONS_USER.format(
    title="Quick Chat", transcript=noisy_transcript
))
parsed = parse_json(resp)
actions = parsed.get("action_items", []) if parsed else []
print(f"  Extracted: {len(actions)} action items")
for a in actions:
    if isinstance(a, dict):
        print(f"    [{a.get('owner','?')}] {a.get('task','?')}")
print("  → Should extract cleanly despite all the filler words")

## 22. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **One prompt for everything** | Categories bleed into each other; some get skipped | Decompose: one prompt per extraction type |
| **Not defining "decision" vs "action item"** | LLM confuses them — lists decisions as actions | Write explicit definitions in system prompt |
| **No owner requirement** | "The team will look into it" is not actionable | Require a named person for each action item |
| **Including implicit commitments** | "We should probably…" is not a commitment | Only extract where someone says "I'll" or is asked and agrees |
| **No evaluation** | No way to know if extraction is improving | Build ground truth and compute precision/recall |
| **Sending full raw transcript** | Wastes tokens on filler words | Parse into turns, optionally clean filler |
| **No chunking strategy** | Long transcripts exceed context window or degrade quality | Map-reduce with merge/dedup step |
| **Mixing meeting types** | A postmortem has different sections than a sprint planning | Adapt prompts per meeting type |

## 23. Production Improvement Ideas

1. **Real-time processing** — Connect to Zoom/Teams transcript API for live summarization
2. **Speaker identification** — Use voice embeddings to identify speakers across meetings
3. **Meeting type detection** — Auto-detect whether it's a standup, planning, review, or postmortem and use tailored prompts
4. **Cross-meeting tracking** — Track action items across multiple meetings (was last week's "I'll do X" actually done?)
5. **Jira/Linear integration** — Auto-create tickets from action items with owner and deadline
6. **Email auto-send** — Generate and send meeting notes to attendees automatically
7. **Custom vocabulary** — Learn team-specific jargon and project names
8. **Sentiment flagging** — Detect tense moments or disagreements for manager awareness

## 24. Exercises

### Exercise 1: Add a "Parking Lot" Extractor

Topics that were raised but deferred ("let's discuss that next time") should go to a "parking lot".

In [ ]:
# ── Exercise 1: Parking lot extraction ────────────────

PARKING_SYSTEM = """You identify deferred topics from meeting transcripts.

A "parking lot" item is a topic that was brought up but explicitly deferred:
- "Let's discuss that next time"
- "We'll table that for now"
- "Add that to the backlog"
- "Not a priority this sprint"

Return JSON: {"parking_lot": ["topic 1", "topic 2"]}
If none, return {"parking_lot": []}"""

PARKING_USER = """Meeting: {title}

Transcript:
{transcript}

Extract deferred topics:"""

# Test on product review (has one deferred topic: design debt sprint)
t = TRANSCRIPTS["product_review"]
resp = ask(PARKING_SYSTEM, PARKING_USER.format(title=t["title"], transcript=t["transcript"]))
parsed = parse_json(resp)
parking = parsed.get("parking_lot", []) if parsed else []

print("PARKING LOT ITEMS:")
for item in parking:
    print(f"  🅿️ {item}")
print("\n→ Should find: 'Design debt sprint' (deferred to Q3)")

### Exercise 2: Meeting Type Adaptation

In [ ]:
# ── Exercise 2: Detect meeting type and adapt extraction ──

DETECT_SYSTEM = """Classify this meeting transcript into one of these types:
- standup: short daily sync, status updates
- sprint_planning: backlog grooming, story points, task assignment
- product_review: metrics, feature assessment, roadmap decisions
- postmortem: incident analysis, root cause, corrective actions
- brainstorm: ideation, creative discussion, no formal decisions
- other: anything else

Return JSON: {"meeting_type": "type", "confidence": "high|medium|low"}"""

for key in TRANSCRIPTS:
    t = TRANSCRIPTS[key]
    resp = ask(DETECT_SYSTEM, f"Transcript:\n{t['transcript'][:500]}")
    parsed = parse_json(resp)
    if parsed:
        print(f"  {t['title']:50s} → {parsed.get('meeting_type', '?')} "
              f"({parsed.get('confidence', '?')})")

print("\nTry adding type-specific prompts: postmortems should extract root cause and timeline.")

### Exercise 3: Cross-meeting Action Tracking

In [ ]:
# ── Exercise 3: Track action items across meetings ────
# Simulate: sprint_planning assigned tasks, postmortem happened later

previous_actions = all_decomp["sprint_planning"].get("action_items", [])
current_transcript = TRANSCRIPTS["postmortem"]["transcript"]

TRACK_SYSTEM = """You check whether action items from a previous meeting were addressed 
in a follow-up meeting. Return JSON with status for each."""

TRACK_USER = """Previous action items:
{previous}

Current meeting transcript:
{transcript}

For each previous action item, determine:
- "completed" if explicitly mentioned as done
- "in_progress" if mentioned but not finished
- "not_mentioned" if not discussed

Return JSON: {{"tracking": [{{"task": "...", "status": "...", "evidence": "..."}}]}}"""

resp = ask(
    TRACK_SYSTEM,
    TRACK_USER.format(
        previous=json.dumps(previous_actions[:4], indent=2),
        transcript=current_transcript,
    ),
)
parsed = parse_json(resp)
tracking = parsed.get("tracking", []) if parsed else []

print("ACTION ITEM TRACKING:")
for item in tracking:
    if isinstance(item, dict):
        status_icon = {"completed": "✅", "in_progress": "🔄", "not_mentioned": "❓"}
        icon = status_icon.get(item.get("status", ""), "❓")
        print(f"  {icon} {item.get('task', '?')[:60]:60s} → {item.get('status', '?')}")

print("\nThis is the foundation for a meeting-to-meeting accountability system.")

### Mini Challenge

1. **Write your own transcript:** Record or write a transcript from a real meeting you attended. Run it through the full pipeline. How accurate are the results?

2. **Prompt ablation:** Remove one element at a time from the action-items prompt (remove the definition of "action item", remove the priority field, remove the "named owner" requirement). Measure how precision and recall change.

3. **Multi-language:** Modify the pipeline to handle transcripts in other languages. Does it work if you just add "The transcript may be in any language. Always output in English." to the system prompt?

4. **Summarization length control:** Instead of "3-5 sentences", make the summary length configurable: one-liner, short (3 sentences), detailed (full paragraph). Compare usefulness.

## 25. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nTranscripts processed: {len(TRANSCRIPTS)}")
print(f"Approaches compared: monolithic vs decomposed (5 prompts)")
print(f"Output formats: Markdown, Slack, JSON")

# Timing comparison
mono_total = sum(all_mono[k].get("_time_sec", 0) for k in TRANSCRIPTS)
decomp_total = sum(all_decomp[k].get("_time_sec", 0) for k in TRANSCRIPTS)
print(f"\nTotal time — Monolithic: {mono_total:.0f}s | Decomposed: {decomp_total:.0f}s")

# Extraction counts
for key in TRANSCRIPTS:
    gt = TRANSCRIPTS[key]["ground_truth"]
    decomp = all_decomp[key]
    print(f"\n{TRANSCRIPTS[key]['title']}:")
    print(f"  Decisions — GT: {len(gt['decisions'])}, Extracted: {len(decomp['decisions'])}")
    print(f"  Actions   — GT: {len(gt['action_items'])}, Extracted: {len(decomp['action_items'])}")

print(f"\nLLM: {LLM_MODEL}")
print(f"Edge cases tested: 3 (implicit actions, disguised decisions, noisy transcript)")

## 26. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Prompt decomposition** beats monolithic prompts — separate calls for summary, decisions, blockers, actions, and next steps |
| 2 | **Define your categories precisely** — "decision" vs "action item" vs "next step" must have clear, written boundaries |
| 3 | **Parse before prompting** — structure the transcript into speaker turns; don't send raw text |
| 4 | **Require named owners** for action items — "someone should" is not actionable |
| 5 | **Evaluate with precision/recall** — ground-truth annotations + fuzzy matching reveals what your pipeline misses |
| 6 | **Chunked map-reduce** handles long transcripts without quality degradation |
| 7 | **Format for the consumer** — Slack, email, and Jira need different output shapes from the same data |
| 8 | **Edge cases matter** — implicit commitments, noisy speech, and disguised decisions test prompt robustness |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*